# Partie B — Cavité entraînée pour $100 \le Re \le 5000$

Énoncé, Partie B : calculer l'écoulement pour plusieurs $Re < 7500$, décrire l'évolution de l'écoulement, comparer à la littérature, tracer $P(Re)$
et identifier une loi d'échelle.

1. Calculs : M1 (projection) et M2 ($\psi$–$\omega$), 11 $Re$ de 100 à 5000, $N = 64$ et $128$, plus $N = 256$ à $Re = 3200$ et $5000$.
2. Évolution de l'écoulement : lignes de courant et centres des tourbillons.
3. Comparaison à la littérature : Ghia et al. (1982) et Erturk et al. (2005), aux mêmes grilles que Ghia.
4. $P(Re)$ et loi d'échelle.

Réglages de la Partie A : grille colocalisée, Euler explicite, advection centrée, gradient conjugué (tol $10^{-8}$), $\Delta t$ = limites de stabilité du
cours. Arrêt quand le résidu stationnaire $\|R\|$ est divisé par $10^3$ depuis $t = 30$ (R sur $u, v$ pour M1, sur $\omega$ pour M2) ; ce critère a été choisi
au cas le plus défavorable, $Re = 5000$ (ADR 0003).

**Les difficultés rencontrées et les diagnostics (critère d'arrêt, fausse solution de M2, erreur de la formule de Thom, erreur propre de Ghia) sont
documentés dans [`Doc/partie_B_limites.md`](../Doc/partie_B_limites.md).** Ce notebook ne garde que les résultats.

In [ ]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"   # défaut de l'environnement CFD sur cette machine (cf. notebook A), avant numpy
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG, RES, REF = ROOT / "figures", ROOT / "results", ROOT / "ref_data"
FIG.mkdir(exist_ok=True)

from cavity.common import solve_poisson
from cavity.projection import run_projection
from cavity.psi_omega import run_psi_omega

RE_LIST = (100, 200, 400, 700, 1000, 1500, 2000, 2500, 3200, 4000, 5000)
RE_GHIA = (100, 400, 1000, 3200, 5000)       # Re publiés par Ghia et al. (1982)
N_GHIA = {re: 128 for re in RE_GHIA} | {5000: 256}   # grille de Ghia : 129² jusqu'à Re = 3200, 257² à Re = 5000
CG_TOL, EPS, T_REF, T_MAX = 1e-8, 1e-3, 30.0, 1000.0
RUNNERS = {"M1": run_projection, "M2": run_psi_omega}
STYLE = {"M1": dict(ls="-", color="C0"), "M2": dict(ls="--", color="C1")}


def run_path(method, re, n):
    return RES / f"B_{method}_Re{re}_N{n}_tol{CG_TOL:.0e}_res{EPS:.0e}_tref{T_REF:g}.npz"


def load_run(path):
    data = np.load(path)
    return {key: (data[key].item() if data[key].ndim == 0 else data[key]) for key in data.files}


def get_run(method, re, n):
    # Relit le calcul s'il existe (faits sur Ruche : scripts/partie_B.py) ; sinon le lance ici (long à haut Re)
    path = run_path(method, re, n)
    if not path.exists():
        run = RUNNERS[method](n, re, "centered", cg_tol=CG_TOL, residual_tol=EPS, t_ref=T_REF, t_max=T_MAX)
        np.savez(path, **{key: np.asarray(val) for key, val in run.items()})
    return load_run(path)


def centerlines(run):
    # u(0.5, y) sur la colonne i = N/2 et v(x, 0.5) sur la ligne j = N/2
    n = run["N"]
    return run["u"][n // 2, :], run["v"][:, n // 2]


def slope(x, y):
    # Pente des moindres carrés, écrite à la main : cov(x, y) / var(x)
    dx = x - x.mean()
    return np.sum(dx * (y - y.mean())) / np.sum(dx * dx)

## 1. Calculs

**Ce qu'on fait.** M1 et M2 depuis le repos jusqu'au stationnaire : 11 $Re$ × 2 méthodes × $N = 64$ et $128$, plus $N = 256$ à $Re = 3200$ et $5000$
(48 calculs, faits sur le calculateur Ruche avec `scripts/partie_B.py` et relus ici).

**Ce qu'on attend.** À haut $Re$, $\Delta t$ est fixé par le schéma centré ($\Delta t \le 2\nu$), indépendamment de $N$ ; l'approche du stationnaire est de plus
en plus lente, donc l'arrêt de plus en plus tardif.

In [ ]:
CASES = [(m, re, n) for re in RE_LIST for n in (64, 128) for m in ("M1", "M2")]
CASES += [(m, re, 256) for re in (3200, 5000) for m in ("M1", "M2")]
runs = {case: get_run(*case) for case in CASES}

print(f"{'Re':>5} {'N':>4} | {'Δt':>8} | {'arrêt M1 (t)':>12} {'M2':>6} | {'calcul M1 (min)':>15} {'M2':>7}")
for re in RE_LIST:
    for n in (64, 128, 256):
        if ("M1", re, n) not in runs:
            continue
        r1, r2 = runs[("M1", re, n)], runs[("M2", re, n)]
        flag = "" if r1["converged"] and r2["converged"] else "  (garde-fou t = 1000 atteint)"
        print(f"{re:5d} {n:4d} | {r1['dt']:8.1e} | {r1['t_final']:12.0f} {r2['t_final']:6.0f} | "
              f"{r1['wall_time'] / 60:15.1f} {r2['wall_time'] / 60:7.1f}{flag}")

**Lecture.**
- L'arrêt est de plus en plus tardif quand $Re$ augmente : $t \approx 50$ à $Re = 400$, $70$–$110$ à $1000$, $210$–$280$ à $3200$, $250$–$430$ à $5000$.
- Le coût suit : quelques minutes aux bas $Re$, jusqu'à 22 h pour M2 à $N = 256$, $Re = 5000$ (Ruche, un cœur par calcul).
- Un seul calcul va jusqu'au garde-fou : M1 à $Re = 100$, $N = 128$, stationnaire avant $t = 30$ (le critère relatif ne peut plus se déclencher) ; son
  résultat est juste.

## 2. Évolution de l'écoulement

**Ce qu'on fait.** Lignes de courant ($N = 128$) pour 6 $Re$. M2 calcule $\psi$ ; pour M1, on la reconstruit par $\Delta\psi = -\omega$ ($\omega$ par différences centrées,
$\psi = 0$ aux parois, même gradient conjugué). Le tourbillon principal tourne dans le sens horaire ($\psi < 0$, en noir) ; les tourbillons secondaires en sens
inverse ($\psi > 0$, en rouge, niveaux logarithmiques car ils sont 100 à $10^5$ fois plus faibles).

**Ce qu'on attend** (Ghia et al., Fig. 3 et Table V) : le tourbillon principal descend vers le centre ; deux tourbillons secondaires dans les coins du bas
(BL, BR) qui grossissent ; un troisième en haut à gauche (TL) qui apparaît entre $Re = 1000$ et $3200$.

In [ ]:
def streamfunction(run):
    # ψ : stockée par M2 ; reconstruite pour M1 (Δψ = -ω, ψ = 0 aux parois)
    if "psi" in run:
        return run["psi"]
    u, v, h = run["u"], run["v"], run["h"]
    omega = ((v[2:, 1:-1] - v[:-2, 1:-1]) - (u[1:-1, 2:] - u[1:-1, :-2])) / (2.0 * h)
    psi, _ = solve_poisson(-omega, h, "dirichlet", tol=1e-12)
    return psi


psis = {key: streamfunction(r) for key, r in runs.items()}
RE_SHOW = (100, 400, 1000, 1500, 3200, 5000)
LEV_NEG = [-0.1175, -0.115, -0.11, -0.10, -0.09, -0.07, -0.05, -0.03, -0.01, -1e-4, -1e-5]
LEV_POS = [1e-8, 1e-7, 1e-6, 1e-5, 5e-5, 1e-4, 2.5e-4, 5e-4, 1e-3, 1.5e-3, 3e-3]
fig, axes = plt.subplots(2, len(RE_SHOW), figsize=(3 * len(RE_SHOW), 6.4))
for row, m in enumerate(("M1", "M2")):
    for col, re in enumerate(RE_SHOW):
        r, ax = runs[(m, re, 128)], axes[row, col]
        X, Y = np.meshgrid(r["x"], r["y"], indexing="ij")
        ax.contour(X, Y, psis[(m, re, 128)], levels=LEV_NEG, colors="k", linewidths=0.6)
        ax.contour(X, Y, psis[(m, re, 128)], levels=LEV_POS, colors="r", linewidths=0.5)
        ax.set_aspect("equal"); ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1])
        ax.set_title(f"{m}, Re = {re}", fontsize=10)
fig.suptitle("Lignes de courant, N = 128 : noir ψ < 0 (tourbillon principal), rouge ψ > 0 (tourbillons secondaires)")
fig.tight_layout()
fig.savefig(FIG / "B_streamlines.png", dpi=150)
plt.show()

# Centres des tourbillons : principal = min de ψ ; secondaires = max de ψ > 1e-7 dans le quart de cavité du coin
# Ghia et al. (1982), Table V : principal (ψ_min, x, y), secondaires (x, y) ; relue sur le PDF (ref_data/primary_vortex.csv)
GHIA_VORTEX = {
    100:  {"principal": (-0.103423, 0.6172, 0.7344)},
    400:  {"principal": (-0.113909, 0.5547, 0.6055), "BL": (0.0508, 0.0469), "BR": (0.8906, 0.1250)},
    1000: {"principal": (-0.117929, 0.5313, 0.5625), "BL": (0.0859, 0.0781), "BR": (0.8594, 0.1094)},
    3200: {"principal": (-0.120377, 0.5165, 0.5469), "BL": (0.0859, 0.1094), "BR": (0.8125, 0.0859), "TL": (0.0547, 0.8984)},
    5000: {"principal": (-0.118966, 0.5117, 0.5352), "BL": (0.0703, 0.1367), "BR": (0.8086, 0.0742), "TL": (0.0625, 0.9102)},
}
CORNERS = {"BL": (0, 0), "BR": (1, 0), "TL": (0, 1)}    # quart de cavité : 0 = moitié basse/gauche, 1 = haute/droite


def vortex_centers(run, psi):
    x, y, half = run["x"], run["y"], run["N"] // 2
    i, j = np.unravel_index(np.argmin(psi), psi.shape)
    out = {"principal": (psi[i, j], x[i], y[j])}
    for name, (sx, sy) in CORNERS.items():
        ii = slice(0, half) if sx == 0 else slice(half, None)
        jj = slice(0, half) if sy == 0 else slice(half, None)
        k, l = np.unravel_index(np.argmax(psi[ii, jj]), psi[ii, jj].shape)
        if psi[ii, jj][k, l] > 1e-7:
            out[name] = (psi[ii, jj][k, l], x[ii][k], y[jj][l])
    return out


print("Tourbillons secondaires présents (N = 128) :")
for re in RE_LIST:
    found = [name for name in CORNERS if name in vortex_centers(runs[("M1", re, 128)], psis[("M1", re, 128)])]
    print(f"   Re = {re:5d} : {', '.join(found) or '—'}")
print("\nCentres, sur la grille de Ghia (N = 128, et N = 256 à Re = 5000) : (x, y), et ψ_min pour le principal")
for re in RE_GHIA:
    n = N_GHIA[re]
    c = {m: vortex_centers(runs[(m, re, n)], psis[(m, re, n)]) for m in ("M1", "M2")}
    for name, g in GHIA_VORTEX[re].items():
        cells = []
        for m in ("M1", "M2"):
            v = c[m].get(name)
            cells.append(f"{m} : absent" if v is None else
                         (f"{m} : ψ = {v[0]:.4f} ({v[1]:.3f}, {v[2]:.3f})" if name == "principal" else f"{m} : ({v[1]:.3f}, {v[2]:.3f})"))
        ghia = f"ψ = {g[0]:.4f} ({g[1]:.3f}, {g[2]:.3f})" if name == "principal" else f"({g[0]:.3f}, {g[1]:.3f})"
        print(f"   Re = {re:4d}, N = {n}, {name:>9} | {cells[0]:32s} | {cells[1]:32s} | Ghia : {ghia}")

**Lecture.** M1 et M2 donnent la même topologie, et son évolution est celle décrite par Ghia :
- le tourbillon principal descend vers le centre : $(0.62, 0.73)$ à $Re = 100$, $(0.52, 0.54)$ à $Re = 5000$ ;
- les tourbillons BL et BR existent dès $Re = 100$ (très faibles) et grossissent avec $Re$ ;
- **TL apparaît entre $Re = 1000$ et $1500$** (Ghia ne donnait que « entre 1000 et 3200 ») ;
- tous les centres sont à 1 ou 2 mailles de ceux de Ghia. Le tourbillon principal de M2 est un peu plus faible que celui de M1 (§3).

## 3. Comparaison à la littérature, aux mêmes grilles que Ghia

**Ce qu'on fait.** Profils $u(0.5, y)$ et $v(x, 0.5)$ sur la grille de Ghia ($N = 128$ jusqu'à $Re = 3200$, $N = 256$ à $Re = 5000$), avec les points de Ghia et
d'Erturk et al. (2005, grille 601², plus précise que Ghia ; publiée pour $Re \ge 1000$). Écart maximal le long de chaque ligne, notre profil étant interpolé
aux points de la référence (points de paroi et coquilles connues de Ghia exclus, `ref_data/README.md`).

**Ce qu'on attend.** Un accord de l'ordre de $10^{-2}$, avec des écarts qui croissent avec $Re$ (couches limites plus minces sur la même grille).

In [ ]:
ghia_u = np.loadtxt(REF / "ghia1982_u_vertical_centerline.csv", delimiter=",")
ghia_v = np.loadtxt(REF / "ghia1982_v_horizontal_centerline.csv", delimiter=",")
erturk_u = np.loadtxt(REF / "erturk2005_u_vertical_centerline.csv", delimiter=",")
erturk_v = np.loadtxt(REF / "erturk2005_v_horizontal_centerline.csv", delimiter=",")
GHIA_COL = {100: 1, 400: 2, 1000: 3, 3200: 4, 5000: 5}
ERTURK_COL = {1000: 1, 5000: 3}
GHIA_TYPOS = {3200: (0.4531, None), 400: (None, 0.9063)}   # coquilles de l'article (u en y, v en x) : points ignorés


def reference(re, source):
    # Points intérieurs (y, u) et (x, v) de la référence ; parois exclues (vitesses imposées)
    if source == "Ghia":
        yu, u, xv, v = ghia_u[:, 0], ghia_u[:, GHIA_COL[re]], ghia_v[:, 0], ghia_v[:, GHIA_COL[re]]
        bad_y, bad_x = GHIA_TYPOS.get(re, (None, None))
        ok_u = (yu > 0) & (yu < 1) & (np.round(yu, 4) != bad_y)
        ok_v = (xv > 0) & (xv < 1) & (np.round(xv, 4) != bad_x)
    else:
        yu, u, xv, v = erturk_u[:, 0], erturk_u[:, ERTURK_COL[re]], erturk_v[:, 0], erturk_v[:, ERTURK_COL[re]]
        ok_u, ok_v = (yu > 0) & (yu < 1), (xv > 0) & (xv < 1)
    return yu[ok_u], u[ok_u], xv[ok_v], v[ok_v]


def max_error(run, re, source):
    yu, u, xv, v = reference(re, source)
    u_c, v_c = centerlines(run)
    return np.abs(np.interp(yu, run["y"], u_c) - u).max(), np.abs(np.interp(xv, run["x"], v_c) - v).max()


fig, axes = plt.subplots(2, len(RE_GHIA), figsize=(4 * len(RE_GHIA), 8))
for col, re in enumerate(RE_GHIA):
    n = N_GHIA[re]
    for m in ("M1", "M2"):
        u_c, v_c = centerlines(runs[(m, re, n)])
        axes[0, col].plot(u_c, runs[(m, re, n)]["y"], **STYLE[m], lw=1.2, label=f"{m}, N = {n}")
        axes[1, col].plot(runs[(m, re, n)]["x"], v_c, **STYLE[m], lw=1.2, label=f"{m}, N = {n}")
    for source, mark in (("Ghia", "ko"), ("Erturk", "k^")):
        if source == "Erturk" and re not in ERTURK_COL:
            continue
        yu, u, xv, v = reference(re, source)
        axes[0, col].plot(u, yu, mark, mfc="none", ms=4, label=source)
        axes[1, col].plot(xv, v, mark, mfc="none", ms=4, label=source)
    axes[0, col].set_title(f"Re = {re}")
    axes[0, col].set_xlabel("$u(0.5, y)$"); axes[0, col].set_ylabel("$y$")
    axes[1, col].set_xlabel("$x$"); axes[1, col].set_ylabel("$v(x, 0.5)$")
for ax in axes.ravel():
    ax.grid(True, alpha=0.3)
axes[0, -1].legend(fontsize=8)
fig.suptitle("Profils sur les lignes médianes, grille de Ghia (N = 128 ; N = 256 à Re = 5000)")
fig.tight_layout()
fig.savefig(FIG / "B_profiles_Ghia.png", dpi=150)
plt.show()

print("Écart maximal sur les lignes médianes, max|Δu| / max|Δv|, sur la grille de Ghia ; ψ_min : écart relatif à Ghia")
print(f"{'Re':>5} {'N':>4} {'méth.':>5} | {'à Ghia':>15} | {'à Erturk':>15} | {'ψ_min':>8} {'écart':>7}")
for re in RE_GHIA:
    n = N_GHIA[re]
    for m in ("M1", "M2"):
        r = runs[(m, re, n)]
        eg = max_error(r, re, "Ghia")
        ee = f"{max_error(r, re, 'Erturk')[0]:6.4f} / {max_error(r, re, 'Erturk')[1]:6.4f}" if re in ERTURK_COL else "–"
        psi_{min} = psis[(m, re, n)].min()
        print(f"{re:5d} {n:4d} {m:>5} | {eg[0]:6.4f} / {eg[1]:6.4f} | {ee:>15} | {psi_{min}:8.4f} "
              f"{100 * (psi_{min} / GHIA_VORTEX[re]['principal'][0] - 1):+6.1f} %")

**Lecture.**
- **Les deux méthodes reproduisent Ghia** sur sa grille : écart maximal $\le 0.025$ jusqu'à $Re = 1000$, $\psi_{\min}$ à moins de 3 %.
- **À haut $Re$, M2 s'écarte plus que M1** : à $Re = 3200$, $N = 128$, $0.060$ contre $0.020$ sur $u$, et $\psi_{\min}$ trop faible de 9 %. Sur la grille fine
  ($N = 256$, $Re = 5000$), M2 rejoint M1. Cause principale : la formule de Thom pour $\omega$ de paroi a une erreur d'ordre 1 qui grandit comme $h\sqrt{Re}$
  (démonstration et tests dans `Doc/partie_B_limites.md` §3).
- **Ghia a sa propre erreur** : à $Re = 5000$, son $\psi_{\min}$ est 2.7 % plus faible que celui d'Erturk. Par rapport à Erturk, M1 est à $0.006$ ($Re = 1000$,
  $N = 128$) et $0.005$ ($Re = 5000$, $N = 256$) : à grille égale, M1 est 3 à 5 fois plus précis que M2.

## 4. $P(Re)$ et loi d'échelle

**Ce qu'on fait.** $P$ stationnaire en fonction de $Re$ (11 points), en log-log, et régression de $\ln P$ en $\ln Re$ (écrite à la main).

**Ce qu'on attend.** Couche limite sous le couvercle d'épaisseur $\delta \sim Re^{-1/2}$, donc $\partial_y u \sim 1/\delta$ et $P \sim \nu/\delta \sim Re^{-1/2}$. La singularité des coins
du couvercle ajoute un terme $\propto \nu \ln N = Re^{-1}\ln N$ (Partie A) : la pente mesurée sera plus forte que $-1/2$ aux $Re$ modérés. On sépare les deux
par $P\sqrt{Re} = a + c\,Re^{-1/2}$, une droite en $Re^{-1/2}$ ($a$ : couche limite, $c$ : coins).

In [ ]:
re_arr = np.array(RE_LIST, float)
P = {(m, n): np.array([runs[(m, re, n)]["P"][-1] for re in RE_LIST]) for m in ("M1", "M2") for n in (64, 128)}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for k, ((m, n), p) in enumerate(P.items()):
    axes[0].loglog(re_arr, p, "o" if n == 128 else "s", color=f"C{k}", ls=STYLE[m]["ls"],
                   label=f"{m}, N = {n} : pente {slope(np.log(re_arr), np.log(p)):.2f}")
    a_c = slope(re_arr**-0.5, p * np.sqrt(re_arr))
    a = (p * np.sqrt(re_arr)).mean() - a_c * (re_arr**-0.5).mean()
    axes[1].plot(re_arr**-0.5, p * np.sqrt(re_arr), "o" if n == 128 else "s", color=f"C{k}",
                 label=f"{m}, N = {n} : a = {a:.2f}, c = {a_c:.2f}")
    axes[1].plot([0, 0.1], [a, a + 0.1 * a_c], color=f"C{k}", lw=0.8, ls=STYLE[m]["ls"])
axes[0].loglog(re_arr, P[("M2", 128)][0] * (re_arr / 100)**-0.5, "k:", label=r"$Re^{-1/2}$")
axes[0].set_xlabel("$Re$"); axes[0].set_ylabel("$P$ stationnaire"); axes[0].set_title("$P(Re)$")
axes[1].set_xlim(0, None)
axes[1].set_xlabel(r"$Re^{-1/2}$"); axes[1].set_ylabel(r"$P\sqrt{Re}$"); axes[1].set_title(r"$P\sqrt{Re} = a + c\,Re^{-1/2}$")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "B_P_Re.png", dpi=150)
plt.show()

p = P[("M1", 128)]
print("Pentes locales d ln P / d ln Re (M1, N = 128) :", np.round(np.diff(np.log(p)) / np.diff(np.log(re_arr)), 2))
p = P[("M2", 128)]
print("Pentes locales d ln P / d ln Re (M2, N = 128) :", np.round(np.diff(np.log(p)) / np.diff(np.log(re_arr)), 2))

**Lecture.**
- **Pente globale** d'environ $-0.7$ sur $100 \le Re \le 5000$, plus forte que $-1/2$ à cause du terme de coin en $Re^{-1}$.
- **Les pentes locales diminuent vers $-1/2$** quand $Re$ augmente : pour M2, de $-0.86$ (entre $Re = 100$ et $200$) à $-0.54$ au-delà de $Re = 2500$ ; pour M1,
  de $-0.87$ à $-0.62$. **Loi d'échelle : $P \propto Re^{-1/2}$ aux grands $Re$** (couche limite du couvercle), avec une correction en $Re^{-1}$ (coins) qui domine
  aux $Re$ modérés. Le palier plus fort de M1 vient de ce terme de coin, plus grand chez M1 ($c$, graphique de droite).
- Le modèle à deux termes est une droite pour les deux méthodes ; $c$ (coins) augmente avec $N$, comme le terme en $\nu\ln N$ trouvé en Partie A. $P$ dépend donc
  de $N$ : on ne compare les $P$ qu'à $N$ fixé.

## Bilan de la Partie B

- **Évolution de l'écoulement** : le tourbillon principal se recentre, les tourbillons de coin BL et BR grossissent, **TL apparaît entre $Re = 1000$ et $1500$** ;
  centres à 1–2 mailles de ceux de Ghia.
- **Comparaison à la littérature** : sur la grille de Ghia, les deux méthodes sont à $\sim 10^{-2}$ de Ghia. M1 est la plus précise (à $0.005$ d'Erturk à
  $Re = 5000$, $N = 256$) ; M2 demande une grille plus fine à haut $Re$, surtout à cause de la formule de Thom.
- **Loi d'échelle** : $P \propto Re^{-1/2}$ aux grands $Re$, plus un terme de coin en $Re^{-1}$.
- **Limites et difficultés** (détail dans `Doc/partie_B_limites.md`) : critère d'arrêt à calibrer à haut $Re$ ; fausse solution de M2 à $N = 64$ pour
  $Re \ge 3200$ ; erreur d'ordre 1 de la formule de Thom ; erreur propre de Ghia (2.7 % à $Re = 5000$).